# Final test: C7, WinCLIP

**CRISP-DM stage:** Evaluation

This is the locked notebook `07_winclip_C7` with **one addition: after half A, half B is scored once**, with the same model and the same benchmark line. Every setting was fixed in the Lock (DECISIONS.md, 17 Nov 2026), and nothing is chosen here.

- **Half A first:** the run repeats the locked half A run, so its numbers can be checked against the saved ones. If they do not repeat, I stop and report before using half B.
- **Half B:** the shared code in `src/final_test.py` measures it with `src/evaluate.py`, saves the numbers in `results/half_b/`, and draws a failure gallery that only describes the result. The cost-based line is applied later, on the laptop, from the saved photo scores.
- WinCLIP has no fitting, so the run repeats exactly.

**Runs on:** Kaggle, GPU.


In [ ]:
!pip install anomalib==2.7.0 open-clip-torch==2.24.0

In [ ]:
CONFIG_ID = "C7"

import csv
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml
from anomalib.data import MVTecAD2
from anomalib.models import WinClip
from anomalib.models.image.winclip import torch_model as winclip_code
from anomalib.models.image.winclip.prompting import create_prompt_ensemble
from torch.utils.data import DataLoader, Subset

USING_GPU = torch.cuda.is_available()
if USING_GPU:
    DEVICE = "cuda"
    INPUT_FOLDER = Path("/kaggle/input")
    EXPECTED_HALF_A_PHOTOS = {"sheet_metal": 54, "vial": 70}
    # Final test: photo counts in half_b_files.csv
    EXPECTED_HALF_B_PHOTOS = {"sheet_metal": 60, "vial": 70}
else:
    # The laptop only proves the code runs. Random weights avoid downloading CLIP (about 800 MB)
    DEVICE = "cpu"
    INPUT_FOLDER = Path("data/sample")
    EXPECTED_HALF_A_PHOTOS = {"sheet_metal": 2, "vial": 4}
    # The laptop sample holds 6 Sheet Metal and 4 Vial photos from half B parts
    EXPECTED_HALF_B_PHOTOS = {"sheet_metal": 6, "vial": 4}
    winclip_code.PRETRAINED = None

# Finds the project code and settings: the private dataset on Kaggle, or the project folder on the laptop
code_folder = None
if USING_GPU:
    for evaluate_file in Path("/kaggle/input").rglob("evaluate.py"):
        code_folder = evaluate_file.parent.parent
else:
    code_folder = Path(".")
if code_folder is None:
    raise FileNotFoundError("Could not find src/evaluate.py")
sys.path.insert(0, str(code_folder))
from src import evaluate
from src import final_test

config_files = list((code_folder / "configs").glob(CONFIG_ID + "_*.yaml"))
with open(config_files[0], encoding="utf-8") as file:
    config = yaml.safe_load(file)
method_settings = config["method"]
SCALES = tuple(method_settings["scales"])
BATCH_SIZE = method_settings["eval_batch_size"]
print("Settings file:", config_files[0].name, "| network:", winclip_code.BACKBONE, "| weights:", winclip_code.PRETRAINED,
      "| scales:", SCALES, "| k_shot:", method_settings["k_shot"])

HALF_A_PARTS = {
    "sheet_metal": {"good": ["001", "002"], "bad": ["000", "004", "005", "007", "009", "012", "013"]},
    "vial": {"good": ["000", "002", "003"], "bad": ["001", "008", "010", "011", "012", "013", "014"]},
}
RESULTS_FOLDER = Path("results")
RESULTS_FOLDER.mkdir(exist_ok=True)

# anomalib's fixed WinCLIP preparation: 240 x 240, bicubic, CLIP's colour numbers
recipe = WinClip.configure_pre_processor().transform
print(recipe)

# The exact sentences used, shown once so nothing is hidden
for product_name in ["sheet_metal", "vial"]:
    normal_prompts, anomalous_prompts = create_prompt_ensemble(product_name)
    print(product_name, "|", len(normal_prompts), "normal sentences, e.g.", normal_prompts[-3], "|",
          len(anomalous_prompts), "anomalous sentences, e.g.", anomalous_prompts[-3])

In [ ]:
def make_loader(dataset, positions=None):
    """Build batches in a fixed order, optionally from chosen photos only."""
    source = dataset
    if positions is not None:
        source = Subset(dataset, positions)
    return DataLoader(source, batch_size=BATCH_SIZE, collate_fn=dataset.collate_fn, shuffle=False)


def score_photos(model, loader):
    """Run WinCLIP on photos. Return maps, photo scores, hottest pixels, masks, paths and model seconds."""
    maps = []
    scores = []
    hottest = []
    masks = []
    paths = []
    model_seconds = 0.0
    with torch.no_grad():
        for batch in loader:
            images, batch_masks = recipe(batch.image, batch.gt_mask)
            images = images.to(DEVICE)
            started = evaluate.start_timer()
            output = model(images)
            model_seconds = model_seconds + evaluate.stop_timer(started)
            # WinCLIP's maps are already [photos, height, width], with no channel to remove
            batch_maps = output.anomaly_map.cpu()
            maps.append(batch_maps)
            scores.append(output.pred_score.reshape(-1).cpu())
            hottest.append(batch_maps.amax(dim=(1, 2)))
            masks.append(batch_masks)
            paths.extend(batch.image_path)
    return torch.cat(maps), torch.cat(scores), torch.cat(hottest), torch.cat(masks), paths, model_seconds

In [ ]:
summary_rows = []
failure_rows = []

for product_name in ["sheet_metal", "vial"]:
    product_folder = None
    for train_folder in INPUT_FOLDER.rglob("train"):
        if train_folder.parent.name == product_name:
            product_folder = train_folder.parent
    datamodule = MVTecAD2(root=product_folder.parent, category=product_name, num_workers=0)
    datamodule.setup()

    half_a_positions = []
    test_paths = datamodule.test_public_data.samples["image_path"].tolist()
    for position in range(len(test_paths)):
        path = Path(test_paths[position])
        if path.stem.split("_")[0] in HALF_A_PARTS[product_name][path.parent.name]:
            half_a_positions.append(position)
    if len(half_a_positions) != EXPECTED_HALF_A_PHOTOS[product_name]:
        raise ValueError("Wrong number of half A photos for " + product_name)

    run_name = CONFIG_ID + "_" + product_name + "_seed42"
    print("===", run_name)
    # class_name is the dataset's category, which is what anomalib uses when none is given
    model = winclip_code.WinClipModel(class_name=datamodule.category, scales=SCALES).to(DEVICE)
    model.eval()
    with torch.no_grad():
        model(torch.zeros(BATCH_SIZE, 3, 240, 240, device=DEVICE))

    validation_maps, validation_scores, ignored_hottest, ignored_masks, ignored_paths, ignored_seconds = score_photos(
        model, make_loader(datamodule.val_data))
    evaluate.reset_peak_memory()
    maps, scores, hottest, masks, paths, model_seconds = score_photos(
        model, make_loader(datamodule.test_public_data, half_a_positions))
    peak_scoring = evaluate.peak_memory_gb()

    labels = []
    for path in paths:
        labels.append(Path(path).parent.name == "bad")
    labels = torch.tensor(labels)

    results = evaluate.evaluate(validation_maps, maps, scores, masks, labels, model_seconds, peak_scoring)
    threshold = results["threshold"]
    results["config"] = CONFIG_ID
    results["product"] = product_name
    results["seed"] = 42
    results["half_a_photos"] = len(paths)
    results["defective_photos"] = int(labels.sum())
    results["good_validation_photos_flagged"] = int((validation_scores > threshold).sum())
    results["good_validation_photos"] = len(validation_scores)
    results["good_half_a_photos_flagged"] = int(((scores > threshold) & ~labels).sum())
    results["defective_half_a_photos_flagged"] = int(((scores > threshold) & labels).sum())
    # Extra check: AU-ROC if the hottest pixel were used as the photo score
    results["image_au_roc_hottest_pixel"] = evaluate.image_auroc(hottest, labels)
    results["fit_seconds"] = 0.0
    results["image_size"] = 240

    with open(RESULTS_FOLDER / (run_name + "_metrics.json"), "w") as file:
        json.dump(results, file, indent=1)
    with open(RESULTS_FOLDER / (run_name + "_photo_scores.csv"), "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["file_name", "label", "score", "above_threshold", "hottest_pixel"])
        for position in range(len(paths)):
            writer.writerow([Path(paths[position]).name, Path(paths[position]).parent.name,
                             round(float(scores[position]), 6), bool(scores[position] > threshold),
                             round(float(hottest[position]), 6)])
    summary_rows.append(results)
    for key in results:
        print(" ", key, results[key])
    # Final test: half B, scored once with the same model and the same benchmark line
    half_b_positions = final_test.half_b_positions(test_paths, product_name, half_a_positions,
                                                   EXPECTED_HALF_B_PHOTOS[product_name])
    evaluate.reset_peak_memory()
    maps_b, scores_b, hottest_b, masks_b, paths_b, seconds_b = score_photos(
    model, make_loader(datamodule.test_public_data, half_b_positions))
    peak_b = evaluate.peak_memory_gb()
    results_b = final_test.save_half_b(RESULTS_FOLDER, run_name, validation_maps, validation_scores,
                                       maps_b, scores_b, masks_b, paths_b, seconds_b, peak_b)
    picks_b = final_test.failure_gallery(
        "gallery_halfB_failures_" + run_name + ".png",
        "MVTec AD 2, " + product_name + " (CC BY-NC-SA 4.0), half B, " + run_name
        + ". Red: mask. Darker blue: more suspicious",
        datamodule.test_public_data, half_b_positions, recipe, maps_b, scores_b, masks_b, paths_b)
    final_test.save_failure_picks(RESULTS_FOLDER, run_name, picks_b)
    # Frees memory before the next run
    del maps_b, masks_b

    # Failure gallery, picked by score as in Phase 6
    defective = []
    good = []
    for position in range(len(paths)):
        if labels[position]:
            defective.append(position)
        else:
            good.append(position)

    def score_of(position):
        return float(scores[position])

    defective.sort(key=score_of)
    good.sort(key=score_of)
    picked = []
    for position in defective[-2:]:
        picked.append(("hit", position))
    for position in defective[:4]:
        picked.append(("hardest defective", position))
    for position in good[-4:]:
        picked.append(("hardest good", position))
    normalize_step = recipe.transforms[1]
    means = torch.tensor(normalize_step.mean).view(3, 1, 1)
    stds = torch.tensor(normalize_step.std).view(3, 1, 1)
    top = float(maps.max())
    bottom = float(maps.min())
    figure, axes = plt.subplots(len(picked), 2, figsize=(8, 3.6 * len(picked)), squeeze=False)
    for row_number in range(len(picked)):
        kind, position = picked[row_number]
        path = Path(paths[position])
        photo = recipe(datamodule.test_public_data[half_a_positions[position]].image)
        photo = (photo * stds + means).clamp(0, 1)
        mask = masks[position].float()
        axes[row_number][0].imshow(photo[0], cmap="gray", vmin=0, vmax=1)
        axes[row_number][1].imshow(maps[position], cmap="Blues", vmin=bottom, vmax=top)
        if bool(mask.any()):
            axes[row_number][0].contour(mask, levels=[0.5], colors="red", linewidths=0.8)
            axes[row_number][1].contour(mask, levels=[0.5], colors="red", linewidths=0.8)
        axes[row_number][0].set_title(kind + ": " + path.parent.name + " " + path.name, fontsize=9)
        axes[row_number][1].set_title("score " + str(round(float(scores[position]), 3)), fontsize=9)
        axes[row_number][0].axis("off")
        axes[row_number][1].axis("off")
        failure_rows.append([CONFIG_ID, product_name, kind, path.parent.name, path.name, round(float(scores[position]), 6)])
    figure.suptitle("MVTec AD 2, " + product_name + " (CC BY-NC-SA 4.0), half A, " + CONFIG_ID
                    + ". Red: mask. Darker blue: more suspicious", y=1.0)
    figure.tight_layout()
    figure.savefig("gallery_failures_" + CONFIG_ID + "_" + product_name + ".png", dpi=120, bbox_inches="tight")
    # Closing without showing keeps dataset images out of the notebook's log
    plt.close(figure)
    with open(RESULTS_FOLDER / (CONFIG_ID + "_failure_picks.csv"), "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["config", "product", "kind", "label", "file_name", "score"])
        for row in failure_rows:
            writer.writerow(row)
    del model
    evaluate.reset_peak_memory()

with open(RESULTS_FOLDER / (CONFIG_ID + "_summary.csv"), "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    for row in summary_rows:
        writer.writerow(row)
print("Finished", CONFIG_ID)